In [241]:
import ast
from typing import Optional, Tuple, List

# --- Kis segédek: kifejezés ekvivalencia, additív lánc, "szorozva 100"-detektálás ---

def ast_equiv(a: ast.AST, b: ast.AST) -> bool:
    """Durva (de praktikus) strukturális ekvivalencia-ellenőrzés néhány fő node-típusra."""
    if type(a) is not type(b):
        return False
    if isinstance(a, ast.Name):
        return a.id == b.id
    if isinstance(a, ast.Constant):
        return a.value == b.value
    if isinstance(a, ast.Subscript):
        return ast_equiv(a.value, b.value) and ast_equiv(a.slice, b.slice)
    if isinstance(a, ast.Index):  # Py<3.9
        return ast_equiv(a.value, b.value)
    if isinstance(a, ast.BinOp) and isinstance(b, ast.BinOp):
        return type(a.op) is type(b.op) and ast_equiv(a.left, b.left) and ast_equiv(a.right, b.right)
    if isinstance(a, ast.UnaryOp) and isinstance(b, ast.UnaryOp):
        return type(a.op) is type(b.op) and ast_equiv(a.operand, b.operand)
    if isinstance(a, ast.Tuple) and isinstance(b, ast.Tuple):
        return len(a.elts) == len(b.elts) and all(ast_equiv(x, y) for x, y in zip(a.elts, b.elts))
    if isinstance(a, ast.Call) and isinstance(b, ast.Call):
        # elég a függvény neve és argumentumok száma + ekvivalencia
        return ast.dump(a.func) == ast.dump(b.func) and len(a.args) == len(b.args) and all(ast_equiv(x,y) for x,y in zip(a.args,b.args))
    # fallback: szigorúbb összevetés
    return ast.dump(a, include_attributes=False) == ast.dump(b, include_attributes=False)

def flatten_add(expr: ast.AST) -> List[ast.AST]:
    """Összeadás bináris fáját listává lapítja (A+B+C+...)."""
    if isinstance(expr, ast.BinOp) and isinstance(expr.op, ast.Add):
        return flatten_add(expr.left) + flatten_add(expr.right)
    return [expr]

def is_const_value(node: ast.AST, val) -> bool:
    return isinstance(node, ast.Constant) and node.value == val

def mul_by_100(expr: ast.AST) -> Tuple[bool, ast.AST]:
    """Ha kifejezés 'X*100' vagy '100*X', visszaadja (True, X)-et; különben (False, expr)."""
    if isinstance(expr, ast.BinOp) and isinstance(expr.op, ast.Mult):
        if is_const_value(expr.left, 100):
            return True, expr.right
        if is_const_value(expr.right, 100):
            return True, expr.left
    return False, expr

# --- Mintakészítő: mindent # helyőrzőre cserél (név, literál, index, hívás) ---

def patternize(node: ast.AST) -> str:
    if isinstance(node, ast.BinOp):
        L = patternize(node.left)
        R = patternize(node.right)
        if isinstance(node.op, ast.Add):
            op = '+'
        elif isinstance(node.op, ast.Sub):
            op = '-'
        elif isinstance(node.op, ast.Mult):
            op = '*'
        elif isinstance(node.op, ast.Div):
            op = '/'
        else:
            op = '?'
        return f"({L}{op}{R})"
    elif isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
        return f"(-{patternize(node.operand)})"
    elif isinstance(node, (ast.Name, ast.Subscript, ast.Call, ast.Constant)):
        return "#"
    elif isinstance(node, ast.Tuple):
        return "(" + ",".join(patternize(e) for e in node.elts) + ")"
    return "?"

# --- Fő kifejezés kiválasztása a return-ből ---

def pick_numeric_expr(ret_value: ast.AST) -> Optional[ast.AST]:
    """A return értékéből kiválasztja az aritmetikai (vagy skalár) kifejezést.
       Tuple esetén az első aritmetikai-nak tűnő elemet preferálja."""
    def first_binop(n: ast.AST) -> Optional[ast.AST]:
        if isinstance(n, ast.BinOp) or isinstance(n, ast.UnaryOp):
            return n
        for child in ast.iter_child_nodes(n):
            fb = first_binop(child)
            if fb is not None:
                return fb
        return None

    if isinstance(ret_value, ast.Tuple):
        # preferáljuk azt az elemet, ami BinOp/UnaryOp; ha nincs, az első elemet
        for e in ret_value.elts:
            fb = first_binop(e)
            if fb is not None:
                return fb
        return ret_value.elts[0] if ret_value.elts else None
    else:
        fb = first_binop(ret_value)
        return fb if fb is not None else ret_value

# --- Magasabb szintű osztályozó ---

def classify(expr: ast.AST) -> str:
    """
    Visszaadja a magasabb szintű művelet nevét:
    'percentage_change', 'percentage', 'average', 'ratio', 'difference', 'sum', 'product', 'negation', 'unknown'
    """
    # 1) Negáció
    if isinstance(expr, ast.UnaryOp) and isinstance(expr.op, ast.USub):
        return "negation"

    # 2) ×100 normalizálása (percentage detektálás miatt)
    times100, base = mul_by_100(expr)
    if times100:
        # ha (X/Y)*100 -> percentage* ; ha ((X - Y)/Y)*100 -> percentage_change
        if isinstance(base, ast.BinOp) and isinstance(base.op, ast.Div):
            num, den = base.left, base.right
            if isinstance(num, ast.BinOp) and isinstance(num.op, ast.Sub) and ast_equiv(num.right, den):
                return "percentage_change"
            return "percentage"

    # 3) Egyszerű hányados: ratio vagy percentage, ha denom 100? (inkább ratio)
    if isinstance(expr, ast.BinOp) and isinstance(expr.op, ast.Div):
        num, den = expr.left, expr.right
        # Átlag: (A + B + ...)/N
        if isinstance(num, ast.BinOp) and isinstance(num.op, ast.Add) and isinstance(den, ast.Constant) and isinstance(den.value, (int, float)):
            terms = flatten_add(num)
            if len(terms) == int(den.value) and den.value > 1:
                return "average"
        # Százalékos változás: (A - B) / B
        if isinstance(num, ast.BinOp) and isinstance(num.op, ast.Sub) and ast_equiv(num.right, den):
            return "percentage_change"  # százalék *nélkül*; ha külön ×100 nincs, akkor arányváltozás alapban
        return "ratio"

    # 4) Összeg / különbség / szorzat
    if isinstance(expr, ast.BinOp):
        if isinstance(expr.op, ast.Add):
            return "sum"
        if isinstance(expr.op, ast.Sub):
            return "difference"
        if isinstance(expr.op, ast.Mult):
            return "product"

    return "unknown"

# --- Driver: egy teljes függvényből minta + címke kinyerése ---

def analyze_function(code: str) -> Tuple[str, str]:
    tree = ast.parse(code)
    # az első return keresése
    class RetVisitor(ast.NodeVisitor):
        def __init__(self):
            self.ret_node = None
        def visit_Return(self, node):
            if self.ret_node is None:
                self.ret_node = node
    rv = RetVisitor()
    rv.visit(tree)
    if rv.ret_node is None:
        return ("", "no_return")

    main_expr = pick_numeric_expr(rv.ret_node.value)
    pat = patternize(main_expr)
    label = classify(main_expr)
    return pat, label


# ---- DEMÓ a te példáddal ----
code = """
def run(value_list):
    deferred_revenue_as_reported = next(item['number_value'] for item in value_list if item['category'] == 'Deferred revenue (2)' and item['header2'] == 'As reported')
    other_non_current_liabilities_as_reported = next(item['number_value'] for item in value_list if item['category'] == 'Other non-current liabilities (2)' and item['header2'] == 'As reported')
    return (deferred_revenue_as_reported - other_non_current_liabilities_as_reported, '')
"""

pattern, label = analyze_function(code)
print(pattern)  # várható: "(#-#)"
print(label)    # várható: "difference"


(#-#)
difference


In [266]:
import ast

def patternize(node):
    if isinstance(node, ast.BinOp):
        left = patternize(node.left)
        right = patternize(node.right)
        op = {ast.Add:"+", ast.Sub:"-", ast.Mult:"*", ast.Div:"/"}.get(type(node.op), "?")
        return f"({left}{op}{right})"
    elif isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id=="round":
        return patternize(node.args[0])  # csak az első argumentum a lényeg
    elif isinstance(node, (ast.Name, ast.Constant, ast.Subscript)):
        return "#"
    return "?"

def classify(pattern: str) -> str:
    if pattern == "(#-#)":
        return "difference"
    if pattern.startswith("((#+#)/#"):
        return "average"
    if ")/#*100" in pattern or ")*100)" in pattern:
        return "percentage"
    return "unknown"

def analyze_function(code: str):
    tree = ast.parse(code)
    assigns = {}
    for node in tree.body[0].body:
        if isinstance(node, ast.Assign):
            assigns[node.targets[0].id] = node.value
        if isinstance(node, ast.Return):
            ret = node.value
            # ha tuple, az első elemet nézzük
            if isinstance(ret, ast.Tuple):
                ret = ret.elts[0]
            # round(...) → bontsuk ki
            if isinstance(ret, ast.Call) and isinstance(ret.func, ast.Name) and ret.func.id=="round":
                ret = ret.args[0]
            # ha változó, helyettesítsük vissza
            if isinstance(ret, ast.Name) and ret.id in assigns:
                ret = assigns[ret.id]
            pat = patternize(ret)
            label = classify(pat)
            return pat, label

# ---- DEMÓ ----
code = """
def run(value_list):
    ebitda_2019 = next(v['number_value'] for v in value_list if v['category'] == 'EBITDA' and v['header2'] == '30 June 2019')
    ebitda_2018 = next(v['number_value'] for v in value_list if v['category'] == 'EBITDA' and v['header2'] == '30 June 2018')
    underlying_ebitda_2019 = next(v['number_value'] for v in value_list if v['category'] == 'Underlying EBITDA' and v['header2'] == '30 June 2019')
    underlying_ebitda_2018 = next(v['number_value'] for v in value_list if v['category'] == 'Underlying EBITDA' and v['header2'] == '30 June 2018')
    
    difference_2019 = ebitda_2019 - underlying_ebitda_2019
    difference_2018 = ebitda_2018 - underlying_ebitda_2018
    
    average_difference = (difference_2019 + difference_2018) / 2
    
    return (round(average_difference, 2), 'thousand')
"""

pattern, label = analyze_function(code)
print(pattern)  # várható: "((#+#)/#)"
print(label)    # várható: "average"


((#+#)/#)
average


In [268]:
import ast

# ---------- segédek

def is_zero(node):
    return isinstance(node, ast.Constant) and node.value in (0, 0.0)

def is_one(node):
    return isinstance(node, ast.Constant) and node.value == 1

def find_first(node, pred):
    """Első részfa, amire pred(node) igaz."""
    if pred(node):
        return node
    for ch in ast.iter_child_nodes(node):
        hit = find_first(ch, pred)
        if hit is not None:
            return hit
    return None

def inline_name(name_node, assigns, depth=5):
    """Változónév visszahelyettesítése az utolsó hozzárendeléssel (max depth)."""
    node = name_node
    seen = set()
    while depth > 0 and isinstance(node, ast.Name) and node.id in assigns and node.id not in seen:
        seen.add(node.id)
        node = assigns[node.id]
        depth -= 1
    return node

def unwrap_numeric(node, assigns):
    """Return-értékből a numerikus kifejezés kibontása (IfExp/Tuple/round/Name inline)."""
    # ternary: vegyük azt az ágat, amiben osztás van; ha egyikben sincs, a "body"-t
    if isinstance(node, ast.IfExp):
        # a tuple első eleme a numerikus rész
        def first_num(n):
            if isinstance(n, ast.Tuple) and n.elts:
                return n.elts[0]
            return n
        body = first_num(node.body)
        orelse = first_num(node.orelse)
        cand = find_first(body, lambda x: isinstance(x, ast.BinOp) and isinstance(x.op, ast.Div)) or \
               find_first(orelse, lambda x: isinstance(x, ast.BinOp) and isinstance(x.op, ast.Div)) or \
               body
        node = cand

    # ha tuple: az első elem a numerikus
    if isinstance(node, ast.Tuple) and node.elts:
        node = node.elts[0]

    # round(x, ...): a szám maga x
    if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id == "round":
        if node.args:
            node = node.args[0]

    # változónév inline
    if isinstance(node, ast.Name):
        node = inline_name(node, assigns)

    return node

def patternize(node, role_map=None):
    """Mintasztring: (#-#), ((sum)/count) stb."""
    role_map = role_map or {}
    if isinstance(node, ast.BinOp):
        op = {ast.Add:"+", ast.Sub:"-", ast.Mult:"*", ast.Div:"/"}.get(type(node.op), "?")
        return f"({patternize(node.left, role_map)}{op}{patternize(node.right, role_map)})"
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
        return f"(-{patternize(node.operand, role_map)})"
    if isinstance(node, ast.Name):
        return role_map.get(node.id, "#")
    if isinstance(node, (ast.Subscript, ast.Call, ast.Constant)):
        return "#"
    return "?"

def classify_general(expr):
    """Alap magasabb szintű címkék, ha nem loop-átlag."""
    if isinstance(expr, ast.BinOp):
        if isinstance(expr.op, ast.Add): return "sum"
        if isinstance(expr.op, ast.Sub): return "difference"
        if isinstance(expr.op, ast.Mult): return "product"
        if isinstance(expr.op, ast.Div):  return "ratio"
    if isinstance(expr, ast.UnaryOp) and isinstance(expr.op, ast.USub):
        return "negation"
    return "unknown"

# ---------- fő elemző

def analyze_function(code: str):
    tree = ast.parse(code)
    func = next((n for n in tree.body if isinstance(n, ast.FunctionDef)), None)
    if not func:
        return "","no_function"

    # hozzárendelések és növelések gyűjtése
    assigns = {}      # utolsó értékadás: var -> expr
    init_vals = {}    # kezdeti érték: var -> expr (első értékadás)
    aug_ops  = {}     # var -> list of (op, right_expr)

    return_node = None

    for node in func.body:
        if isinstance(node, ast.Assign) and len(node.targets)==1 and isinstance(node.targets[0], ast.Name):
            var = node.targets[0].id
            if var not in init_vals:
                init_vals[var] = node.value
            assigns[var] = node.value
        elif isinstance(node, ast.AugAssign) and isinstance(node.target, ast.Name):
            var = node.target.id
            aug_ops.setdefault(var, []).append((type(node.op), node.value))
        elif isinstance(node, ast.For):
            # belső növelések a for-ban is
            for sub in ast.walk(node):
                if isinstance(sub, ast.AugAssign) and isinstance(sub.target, ast.Name):
                    var = sub.target.id
                    aug_ops.setdefault(var, []).append((type(sub.op), sub.value))
        elif isinstance(node, ast.Return):
            return_node = node.value

    if return_node is None:
        return "","no_return"

    # Azonosítsuk a sum és count változókat heurisztikával:
    sum_vars = set()
    count_vars = set()
    for var, ops in aug_ops.items():
        # összeadó változó: kezdet 0, és van += valami (nem feltétlen 1)
        if var in init_vals and is_zero(init_vals[var]) and any(op is ast.Add for op,_ in ops):
            sum_vars.add(var)
        # darabszám: kezdet 0, és legalább egy += 1
        if var in init_vals and is_zero(init_vals[var]) and any(op is ast.Add and is_one(rhs) for op,rhs in ops):
            count_vars.add(var)

    # numerikus kifejezés kibontása
    expr = unwrap_numeric(return_node, assigns)

    # loop-átlag detektálás: total/count mintázat
    role_map = {}
    if isinstance(expr, ast.BinOp) and isinstance(expr.op, ast.Div):
        L, R = expr.left, expr.right
        # inline mindkettő a biztonság kedvéért
        if isinstance(L, ast.Name): L = inline_name(L, assigns)
        if isinstance(R, ast.Name): R = inline_name(R, assigns)

        def name_of(n):
            return n.id if isinstance(n, ast.Name) else None

        lname, rname = name_of(L), name_of(R)
        if lname in sum_vars and rname in count_vars:
            role_map = {lname:"sum", rname:"count"}
            return patternize(expr, role_map), "average"
        # Néha a bal oldal maga egy összeadás-lánc Name helyett; ettől még átlag lehet,
        # de ha nincs egyértelmű sum/count név, esünk a generálba.

    # ha nem loop-átlag, általános minta + címke
    pat = patternize(expr, {})
    label = classify_general(expr)
    return pat, label


# ---------- DEMÓ a példáddal
code = """
def run(value_list):
    total = 0.0
    count = 0
    for item in value_list:
        if item['category'] == 'Defined contribution schemes' and item['header1'] == '2019 â‚¬ million':
            total += item['number_value']
            count += 1
    return (total / count, 'million') if count > 0 else (0.0, '')
"""

pattern, label = analyze_function(code)
print(pattern)  # -> "((sum)/(count))"
print(label)    # -> "average"


(#/#)
ratio


In [263]:
errors = pd.read_csv('res/e38_18r.csv').query('exact_match == False')

In [264]:
code = """
def run(value_list):
    deferred_revenue_as_reported = next(item['number_value'] for item in value_list if item['category'] == 'Deferred revenue (2)' and item['header2'] == 'As reported')
    other_non_current_liabilities_as_reported = next(item['number_value'] for item in value_list if item['category'] == 'Other non-current liabilities (2)' and item['header2'] == 'As reported')
    return (deferred_revenue_as_reported - other_non_current_liabilities_as_reported, '')
"""
analyze_function(code)

('(#-#)', 'difference')

In [293]:
res_ast=[]
for i, item in errors.iterrows():  
    #print (item["question"], item["calc_pattern"])
    #print(item["code"])
    try:
        if item["code"] and type(item["code"]) == str:
            pattern, label =  analyze_function(item["code"])
        else:
            pattern = "no_code"
        if pattern.startswith("(") and pattern.endswith(")"):
            pattern = pattern[1:-1]
        res_ast.append(pattern)
    except Exception as e:
        print(e)
        print(type(item["code"]))


In [327]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
from pathlib import Path
import textwrap
import cga_utils
import math
from typing import Iterable, List, Optional, Tuple
import hdbscan
from sklearn.feature_extraction import DictVectorizer
EPS = 1e-9


error_summaries=pd.read_csv('res/error_summary_e38_v18r.csv')
errors["error_summary"]=error_summaries["problem"]
error_summaries2=pd.read_csv('res/error_summary2_e38_v18r.csv')
errors["fix_summary"]=error_summaries2["summary"]
errors["pred_ast"]=res_ast


def item_to_onehot(_list, item):
    mapping = {p: i for i, p in enumerate(_list)}
    v = np.zeros(len(_list) + 1) #+other
    if item in mapping:
        v[mapping[item]] = 1
    else:
        v[-1] = 1
    return v

patterns = set(cga_utils.aggregate_calc_patterns(errors['calc_pattern'], other_group_size=1, merge_additions=False))
errors["calc_pattern_oh"] = errors["calc_pattern"].apply(lambda pattern: item_to_onehot(patterns, pattern))

error_codes = set(cga_utils.aggregate_calc_patterns(errors['error_code'], other_group_size=0, merge_additions=False))
errors["error_code_oh"] = errors["error_code"].apply(lambda code: item_to_onehot(error_codes, code))

asts = set(cga_utils.aggregate_calc_patterns(errors['pred_ast'], other_group_size=1, merge_additions=False))
errors["ast_oh"] = errors["pred_ast"].apply(lambda pattern: item_to_onehot(asts, pattern))

scales = set(errors['scale'])
errors["scale_oh"] = errors["scale"].apply(lambda pattern: item_to_onehot(scales, pattern))
errors["pred_scale_oh"] = errors["pred_scale"].apply(lambda pattern: item_to_onehot(scales, pattern))

#model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
#embeddings = model.encode(questions)

errors["rel_err"] = errors.apply(lambda row: (row["pred"] - row["answer"]) / max(row["answer"], EPS), axis=1  )
errors["abs_err"] = errors.apply(lambda row: row["pred"] - row["answer"], axis=1  )

errors["magnitude_bucket"] = errors.apply(lambda row: int(math.floor(math.log10(abs(row["answer"])+EPS))) if row["answer"] != 0 else -1, axis=1  )

def err_bucket(rel_err):
    if rel_err < 1e-6: 
        return 0
    elif rel_err < 0.01:
        return 1
    elif rel_err < 0.1:
        return 2
    elif rel_err < 1:
        return 3
    else:
        return 4

errors["rel_error_bucket"] = errors["rel_err"].apply(err_bucket)


#    sign_err = int((y_pred * y_true) < 0)

errors["ratio"] = errors.apply(lambda row: (row["pred"] / (row["answer"]+EPS)) if row["answer"] != 0 else np.inf,  axis=1 )
errors["x100_flag"] = errors["ratio"].apply(lambda ratio: int(0.95 < ratio/100 < 1.05 or 0.95 < ratio*100 < 1.05))
errors["has_error_text"] = errors["error_text"].apply(lambda txt: txt != None and txt != '')


errors["ratio_is_inf"] = np.isinf(errors["ratio"]).astype(int)
errors["ratio"] = np.where(np.isinf(errors["ratio"]), np.nan, errors["ratio"])

# 2) Erősen ferde oszlopok log1p-vel és/vagy winsorize/clip
for col in ["abs_err", "rel_err", "ratio"]:
    if col in errors:
        # negatív is lehet -> signed log1p
        errors[col] = np.sign(errors[col]) * np.log1p(np.abs(errors[col]))

# 3) NaN-ek kezelése (pl. median impute)
num_cols = ["rel_err","abs_err","ratio"]
for col in num_cols:
    if col in errors:
        med = errors[col].median()
        errors[col] = errors[col].fillna(med)

# 4) Skálázás a folytonosokra (RobustScaler a kilógók ellen)
from sklearn.preprocessing import RobustScaler
cont = errors[num_cols].values
scaler = RobustScaler().fit(cont)
errors[num_cols] = scaler.transform(cont)

errors["scale"] = errors["scale"].fillna("")
errors["pred_scale"] = errors["pred_scale"].fillna("")

#vects = errors.drop(["ts", "qid", "pred_scale", "question", "derivation", "calc_pattern", "pred", "answer", "scale", "value_match", "error_text",	"value_list", "code",	"selected_values","needed_values","exact_match",	"error_code"], axis=1)
#vects = errors[["calc_pattern", "error_code",  "pred_ast", "selection_success", "sign_error", "is_parenth_in_table", "has_code_abs",
# "scale", "pred_scale",  "rel_error_bucket", "magnitude_bucket", "x100_flag",  "has_error_text"]]
vects = errors[["calc_pattern", "pred_ast", "scale", "pred_scale"]]

#vects =vects.drop(vects.columns[0], axis=1)
X_dict = vects.to_dict(orient="records")
dv = DictVectorizer(sparse=False)
X_flags = dv.fit_transform(X_dict)

#np.isfinite(X_flags).all()

def cluster_hdbscan(X: np.ndarray, min_cluster_size: int = 8, min_samples: int = 2) -> Tuple[np.ndarray, Optional[float]]:
    if hdbscan is None:
        print("[warn] hdbscan not installed; falling back to agglomerative.")
        return cluster_agglomerative(X, min_cluster_size)
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples, metric="manhattan")
    labels = clusterer.fit_predict(X)
    return labels, None
    from typing import Iterable, List, Optional, Tuple
    
labels, thr = cluster_hdbscan(X_flags, min_cluster_size=4, min_samples=1)

In [328]:
(labels == -1).mean()

0.23529411764705882

In [329]:
labels

array([ 7,  7, -1,  7, -1, -1,  1, -1,  1,  1,  9, 17, 12, 10, -1,  5,  5,
       -1,  2, -1, -1, -1, 16, -1,  3,  3, 16, 16, 11, 16, -1, -1, 15, 15,
       13,  4, 16,  4, 13, 11,  6, 11, 17,  0,  0,  9, 16, -1,  4,  9,  9,
        8,  3,  3,  4, -1, -1, -1, 10, 12, 16,  8, -1, -1,  9,  9, -1,  0,
       11, -1, -1, -1,  8,  0,  0,  0, -1, 14,  2, 13,  1,  1,  3, 17,  5,
       11, -1, -1, 10, 10, -1, -1, 17,  8, 11, 11, 16, -1,  9, 17, 16, 16,
        2,  2,  1,  1,  1, -1, 16, 16, -1, 16,  1, -1,  1, 16, 12, 12, 12,
       15, -1,  9, 16, 12, -1, 16, -1,  9,  7,  7,  9, 10, 10, -1, 11, -1,
        0, -1, -1,  6, 16, 17, 14, 10,  7,  4,  0,  0, 11, -1, -1, 16, 16,
       15, -1,  3,  4, 13, 12,  4, 11,  1, 14,  0,  6,  6, -1,  4,  3, 10,
       10, 13,  5, 11,  7, 11, 14, 17,  8, 16,  4,  3, -1, -1, 13, 13, 13])

In [330]:
errors["labels"] = labels

In [331]:
errors["labels"].value_counts()

labels
-1     44
 16    20
 11    12
 1     11
 9     10
 0     10
 4      9
 10     9
 3      8
 13     8
 7      7
 12     7
 17     7
 8      5
 2      4
 15     4
 5      4
 6      4
 14     4
Name: count, dtype: int64

In [342]:
[(i["question"], i["calc_pattern"], i["pred_ast"]) for idx, i in errors.iterrows() if i["labels"] == 7]

[('What is the 2019 average defined contribution schemes?', '(#+#)/#', '#/#'),
 ('What is the 2019 average defined benefit schemes?', '(#+#)/#', '#/#'),
 ('What is the 2019 average free cash flow?', '(#+#)/#', '#'),
 ('What is the 2019 average total amount falling due within one year?',
  '(#+#)/#',
  '#'),
 ('What is the 2019 average total amount falling due after more than one year?',
  '(#+#)/#',
  '#'),
 ('What is the average unvested restricted stock?', '(#+#)/#', '#/#'),
 ('What was the total non-USD denominated monetary liabilities as at 31 December 2019?',
  '#+#',
  '#')]

In [326]:
def try_hdbscan(X, mcs_list=(2,3,4, 5,6, 10,15), ms_list=(1,2,3, 4, 5)):
    from collections import defaultdict
    out = []
    for mcs in mcs_list:
        for ms in ms_list:
            cl = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=ms, metric="manhattan")
            labels = cl.fit_predict(X)
            n_clusters = len(set(labels) - {-1})
            noise_ratio = (labels == -1).mean()
            out.append((mcs, ms, n_clusters, noise_ratio))
    return out

results = try_hdbscan(X_flags)
# nézd: klaszterszám és zajarányszám hogyan változik
pd.DataFrame(try_hdbscan(X_flags)).sort_values(by=3)

,0,1,2,3
0,2,1,46,0.117647
15,5,1,11,0.203209
5,3,1,27,0.219251
30,15,1,3,0.224599
26,10,2,4,0.224599
20,6,1,10,0.229947
10,4,1,18,0.235294
28,10,4,3,0.251337
33,15,4,3,0.251337
1,2,2,31,0.267380


In [221]:
len(errors)

187

In [75]:
import numpy as np
import pandas as pd

# csak a numerikus oszlopokkal dolgozz
num = vects.select_dtypes(include=[np.number])

# boolean maszk az ±inf értékekre
mask_inf = np.isinf(num.to_numpy())

# 1) mely OSZLOPOK tartalmaznak inf-et?
cols_with_inf = num.columns[mask_inf.any(axis=0)].tolist()

# 2) mely SOROK tartalmaznak inf-et?
rows_with_inf = num.index[mask_inf.any(axis=1)].tolist()

# 3) pontos CELLÁK (sorindex, oszlopnév) ahol inf van
cells_with_inf = [(num.index[i], num.columns[j]) for i, j in zip(*np.where(mask_inf))]

# 4) gyors számlálás: hány inf oszloponként
inf_count_per_col = pd.Series(mask_inf.sum(axis=0), index=num.columns).sort_values(ascending=False)

cols_with_inf, rows_with_inf[:10], cells_with_inf[:10], inf_count_per_col.head()


([],
 [],
 [],
 magnitude_bucket    0
 x100_flag           0
 dtype: int64)

In [74]:
errors["x100_flag"].min()

0

In [52]:
errors

,Unnamed: 0,ts,qid,question,derivation,calc_pattern,pred,pred_scale,answer,scale,...,error_code,rel_err,abs_err,magnitude_bucket,rel_error_bucket,ratio,x100_flag,has_error_text,ratio_is_inf,labels
5,5,1.755582e+09,a0414f81-8dc2-44b2-a441-2c9d9c805c4d,What is the 2019 average defined contribution ...,(166+178)/2,(#+#)/#,166.00,million,172.00,million,...,selection_error,-0.246587,-0.987278,2,0,-6.243776e-02,0,True,0,5
6,6,1.755582e+09,bf7abd62-d9cd-48d2-8826-1457684019a3,What is the 2019 average defined benefit schemes?,(57+44)/2,(#+#)/#,57.00,million,50.50,million,...,selection_error,0.870721,1.022282,1,3,2.213183e-01,0,True,0,5
7,7,1.755582e+09,4d259081-6da6-44bd-8830-e4de0031744c,What is the difference between 2019 average de...,[(166+178)/2] - [(57+44)/2],[(#+#)/#]-[(#+#)/#],109.00,million,121.50,million,...,selection_error,-0.704223,-1.320502,2,0,-1.873959e-01,0,True,0,5
14,14,1.755582e+09,dc5e217a-a7b3-4fc9-ac0f-13d328f26b20,What is the 2019 average free cash flow?,"(4,411+4,044)/2",(#+#)/#,4411.00,million,4227.50,million,...,selection_error,0.305568,2.647229,3,2,7.618878e-02,0,True,0,5
15,15,1.755582e+09,7cd3aedf-1291-4fea-bc9d-a25c65727b7b,What is the 2018 average free cash flow?,"(4,044+3,316)/2",(#+#)/#,4044.00,million,3680.00,million,...,selection_error,0.678305,2.993375,3,2,1.712925e-01,0,True,0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
480,480,1.755583e+09,890e40b2-3009-449f-b806-49d0a4fa82cd,What is the percentage change in the total bal...,"(3,933.5-2,623.7)/2,623.7",(#-#)/#,49.92,NaN,49.92,percent,...,scale_error,0.000000,0.000000,1,0,7.732285e-11,0,True,0,-1
484,484,1.755583e+09,6ce427fa-d1ff-481d-b6ba-7950d43e7c22,What is the proportion of long-term debt payme...,"(2,747.6+2,287.0)/10,556.6",(#+#)/#,0.48,million,0.48,NaN,...,scale_error,0.000000,0.000000,-1,0,-3.583443e-09,0,True,0,-1
489,489,1.755583e+09,5b48fea5-61de-401b-aea9-b2b90b7a0eeb,What is the percentage of non-vested shares gr...,"473,550/1,283,281",#/#,36.90,NaN,36.90,percent,...,scale_error,0.000000,0.000000,1,0,6.478239e-11,0,True,0,-1
490,490,1.755583e+09,8a7ef462-5a25-48c9-8a49-3fb543a73785,What is the percentage of non-vested shares ve...,"365,223/1,283,281",#/#,-40.34,NaN,28.46,percent,...,calculation_error,-8.837442,-2.154067,1,0,-5.591837e+00,0,True,0,-1
